# 01 — EDA & chia tập BRSET

**Cell 1 → Cell 4**: nạp metadata, phân bố 5 mức độ, kiểm tra quan hệ bệnh nhân ↔ 2 mắt,
thống kê kích thước ảnh và sinh 3 file split `train/val/test` theo **patient-level**.

> Chạy notebook từ thư mục gốc của repo để `src` import được.

## Cell 1 — Thiết lập môi trường

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from src.utils.config import load_project_configs
from src.data import (
    assert_no_leakage,
    describe_splits,
    patient_level_split,
    save_splits,
)

configs = load_project_configs()
data_cfg = configs["data"]
CLASS_NAMES = data_cfg["classes"]["names"]

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Project root:", PROJECT_ROOT)
print("Classes:", CLASS_NAMES)

## Cell 2 — Nạp metadata & phân bố nhãn

Sửa `LABELS_CSV` và phần `rename` cho khớp với bản BRSET bạn tải.
Ba cột bắt buộc sau khi chuẩn hoá: `image_path`, `patient_id`, `label`.

In [ ]:
# TODO: đổi đường dẫn / tên cột theo file metadata thực tế của BRSET
LABELS_CSV = PROJECT_ROOT / "data/raw/labels.csv"
df = pd.read_csv(LABELS_CSV)

# Ví dụ chuẩn hoá tên cột (bỏ comment và sửa cho đúng):
# df = df.rename(columns={"image": "image_path", "patient": "patient_id", "dr_grade": "label"})

required = {"image_path", "patient_id", "label"}
missing = required - set(df.columns)
assert not missing, f"Thiếu cột: {missing}"

df["label"] = df["label"].astype(int)
print(f"Số ảnh: {len(df):,} | Số bệnh nhân: {df['patient_id'].nunique():,}")
display(df.head())

counts = (
    df["label"]
    .map(dict(enumerate(CLASS_NAMES)))
    .value_counts()
    .reindex(CLASS_NAMES)
)
percent = (counts / counts.sum() * 100).round(1)
summary = pd.DataFrame({"images": counts, "percent": percent})
display(summary)

ax = counts.plot(kind="bar", color=sns.color_palette("viridis", len(counts)), figsize=(8, 4))
ax.set_title("Phân bố 5 mức độ bệnh võng mạc đái tháo đường")
ax.set_xlabel("Mức độ")
ax.set_ylabel("Số ảnh")
for index, value in enumerate(counts.values):
    ax.text(index, value, f"{value:,}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Cell 3 — Kiểm tra quan hệ bệnh nhân ↔ 2 mắt

Mục tiêu: chứng minh vì sao **phải** chia tập theo `patient_id`.
Nếu một bệnh nhân xuất hiện ở cả train và test, mô hình sẽ học thuộc bệnh nhân
thay vì học tổn thương → điểm test bị thổi phồng.

In [ ]:
eyes_per_patient = df.groupby("patient_id").size()
print("Số ảnh / bệnh nhân:")
print(eyes_per_patient.value_counts().sort_index().to_string())

both_eyes = (eyes_per_patient >= 2).sum()
print(f"\nBệnh nhân có >= 2 ảnh: {both_eyes:,} / {eyes_per_patient.size:,}")

inconsistent = (
    df.groupby("patient_id")["label"].nunique().gt(1).sum()
)
print(f"Bệnh nhân có nhãn không nhất quán giữa các ảnh: {inconsistent:,}")

duplicate_rows = df.duplicated(subset=["image_path"]).sum()
print(f"Đường dẫn ảnh bị trùng: {duplicate_rows:,}")

## Cell 4 — Mẫu ảnh & kích thước gốc

Kiểm tra trực quan chất lượng ảnh và sự khác nhau về kích thước/khung đen
— đây là lý do cần bước ROI crop + Resize 384×384.

In [ ]:
def resolve(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return path
    for base in (PROJECT_ROOT / "data/processed", PROJECT_ROOT / "data/raw", PROJECT_ROOT):
        candidate = base / path
        if candidate.is_file():
            return candidate
    return PROJECT_ROOT / path


sample = df.sample(min(12, len(df)), random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for axis, (_, row) in zip(axes.ravel(), sample.iterrows()):
    with Image.open(resolve(row["image_path"])) as handle:
        axis.imshow(np.asarray(handle.convert("RGB")))
    axis.set_title(f"{CLASS_NAMES[int(row['label'])]} — {row['patient_id']}", fontsize=9)
    axis.axis("off")
for axis in axes.ravel()[len(sample):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

sizes = []
for path_value in df["image_path"].head(300):
    with Image.open(resolve(path_value)) as handle:
        sizes.append(handle.size)
size_df = pd.DataFrame(sizes, columns=["width", "height"])
print(size_df.describe().round(1))
print("\nSố tổ hợp kích thước khác nhau:", size_df.drop_duplicates().shape[0])

## Cell 5 — Chia tập theo bệnh nhân & lưu CSV

Tỉ lệ lấy từ `configs/data_config.yaml` (mặc định 70/15/15, seed 42).
Sau khi chia, `assert_no_leakage` sẽ raise nếu có bệnh nhân nằm ở 2 tập.

In [ ]:
split_cfg = data_cfg["splits"]

train_df, val_df, test_df = patient_level_split(
    df,
    val_size=split_cfg["val_size"],
    test_size=split_cfg["test_size"],
    seed=split_cfg["seed"],
    patient_column=split_cfg["patient_column"],
    label_column=split_cfg["label_column"],
)
assert_no_leakage(
    train_df, val_df, test_df, patient_column=split_cfg["patient_column"]
)

splits = {"train": train_df, "val": val_df, "test": test_df}
display(describe_splits(splits, class_names=CLASS_NAMES))

created = save_splits(
    splits,
    out_dir=PROJECT_ROOT / "data/splits",
    filenames={
        "train": split_cfg["train_csv"],
        "val": split_cfg["val_csv"],
        "test": split_cfg["test_csv"],
    },
)
for name, path in created.items():
    print(f"{name}: {path}")